# AMD Hackathon — Vulnerability-Aware Java LLM

Script 3: Fine-tune Qwen3-8B on our cleaned dataset using Unsloth

Two training modes in this file:
  MODE A — SFT  (Supervised Fine-Tuning)   → stable, recommended first run
  MODE B — GRPO (Reinforcement Learning)   → reward-based, for precision improvement

Run:
  pip install unsloth vllm
  python 03_unsloth_train.py --mode sft
  python 03_unsloth_train.py --mode grpo

Dataset expected at: output/train_chat_template.jsonl
  (generated by 02_clean_and_convert.py)

Format of each JSONL line:
  {
    "messages": [
      {"role": "system",    "content": "..."},
      {"role": "user",      "content": "## Security Vulnerability Report\n..."},
      {"role": "assistant", "content": "```java\n...\n```"}
    ]
  }

In [ ]:
from __future__ import annotations
import argparse
import json
import os
import re
import sys
from pathlib import Path

## Config — edit these before running

In [ ]:
# ── Configuration — edit these before running ─────────────────────────────────

MODEL_NAME     = "unsloth/Qwen3-8B-unsloth-bnb-4bit"   # change to Qwen3-14B later
DATASET_PATH   = "output/train_chat_template.jsonl"
OUTPUT_DIR     = "output/qwen3-8b-java-vuln-lora"
MAX_SEQ_LENGTH = 4096          # covers most Java fix pairs; bump to 8192 if OOM is fine
LOAD_IN_4BIT   = True          # QLoRA — set False for 16-bit LoRA (needs more VRAM)
LORA_RANK      = 32            # 16–64 typical; higher = more capacity, more VRAM
LORA_ALPHA     = 32            # usually = lora_rank
LORA_DROPOUT   = 0.05
TARGET_MODULES = [             # Qwen3 attention + MLP layers
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]

# SFT trainer settings
SFT_BATCH_SIZE    = 2
SFT_GRAD_ACCUM    = 4          # effective batch = 2 * 4 = 8
SFT_EPOCHS        = 3
SFT_LR            = 2e-4
SFT_WARMUP_RATIO  = 0.05
SFT_MAX_STEPS     = -1         # -1 = use epochs; set e.g. 300 for a quick test run

# GRPO settings
GRPO_BATCH_SIZE        = 1     # per device
GRPO_GRAD_ACCUM        = 4
GRPO_MAX_STEPS         = 500   # minimum 300; more = better
GRPO_LR                = 5e-6
GRPO_NUM_GENERATIONS   = 6     # samples per prompt; 4–8 typical; lower = less VRAM
GRPO_MAX_NEW_TOKENS    = 1024  # max tokens the model generates per response
GRPO_TEMPERATURE       = 0.8

# ── Run settings ───────────────────────────────────────────────────────────────
# Training mode: "sft" (Supervised Fine-Tuning, stable) or "grpo" (RL, advanced)
TRAINING_MODE = "sft"

# Set True to merge LoRA into the base model and save as float16 (needed for vLLM)
SAVE_MERGED = False

# HuggingFace Hub repo to push the adapter to (e.g. "myuser/qwen3-java-vuln")
# Leave as "" to skip. Requires HF_TOKEN environment variable.
PUSH_TO_HUB = ""

## Load dataset from JSONL

In [ ]:
def load_jsonl_dataset(path: str):
    """
    Load our train_chat_template.jsonl into a HuggingFace Dataset.
    Each line: {"messages": [{"role": ..., "content": ...}, ...]}
    """
    from datasets import Dataset

    records = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))

    print(f"Loaded {len(records)} records from {path}")
    return Dataset.from_list(records)

## MODE A: SFT — Supervised Fine-Tuning

In [ ]:
def run_sft():
    print("\n" + "="*60)
    print("MODE: Supervised Fine-Tuning (SFT)")
    print("="*60)

    # ── 1. Load model + tokenizer ────────────────────────────────────────────
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = None,           # auto-detect: float16 or bfloat16
        load_in_4bit   = LOAD_IN_4BIT,
    )

    # Apply Qwen3 chat template to tokenizer
    tokenizer = get_chat_template(
        tokenizer,
        chat_template = "qwen-2.5",      # Qwen3 uses the qwen-2.5 template key
    )

    # ── 2. Add LoRA adapters ─────────────────────────────────────────────────
    model = FastLanguageModel.get_peft_model(
        model,
        r                  = LORA_RANK,
        target_modules     = TARGET_MODULES,
        lora_alpha         = LORA_ALPHA,
        lora_dropout       = LORA_DROPOUT,
        bias               = "none",
        use_gradient_checkpointing = "unsloth",   # Unsloth's memory-efficient GC
        random_state       = 42,
        use_rslora         = True,         # RSLoRA: normalises adapters, more stable
        loftq_config       = None,
    )

    print(model.print_trainable_parameters())

    # ── 3. Load and format dataset ───────────────────────────────────────────
    dataset = load_jsonl_dataset(DATASET_PATH)

    def formatting_prompts_func(examples):
        """Apply the tokenizer's chat template to each messages list."""
        texts = [
            tokenizer.apply_chat_template(
                msgs,
                tokenize            = False,
                add_generation_prompt = False,
            )
            for msgs in examples["messages"]
        ]
        return {"text": texts}

    dataset = dataset.map(formatting_prompts_func, batched=True)

    # Optional: split off 5% for evaluation
    split = dataset.train_test_split(test_size=0.05, seed=42)
    train_dataset = split["train"]
    eval_dataset  = split["test"]
    print(f"Train: {len(train_dataset)} | Eval: {len(eval_dataset)}")

    # ── 4. SFT Trainer ───────────────────────────────────────────────────────
    from trl import SFTTrainer
    from transformers import TrainingArguments
    from unsloth import is_bfloat16_supported

    training_args = TrainingArguments(
        per_device_train_batch_size = SFT_BATCH_SIZE,
        gradient_accumulation_steps = SFT_GRAD_ACCUM,
        num_train_epochs            = SFT_EPOCHS if SFT_MAX_STEPS == -1 else None,
        max_steps                   = SFT_MAX_STEPS if SFT_MAX_STEPS > 0 else -1,
        learning_rate               = SFT_LR,
        warmup_ratio                = SFT_WARMUP_RATIO,
        lr_scheduler_type           = "cosine",
        fp16                        = not is_bfloat16_supported(),
        bf16                        = is_bfloat16_supported(),
        logging_steps               = 10,
        save_steps                  = 100,
        save_total_limit            = 3,
        eval_strategy               = "steps",
        eval_steps                  = 100,
        output_dir                  = OUTPUT_DIR,
        optim                       = "adamw_8bit",
        weight_decay                = 0.01,
        seed                        = 42,
        report_to                   = "none",    # set "wandb" if you want W&B logging
    )

    trainer = SFTTrainer(
        model           = model,
        tokenizer       = tokenizer,
        train_dataset   = train_dataset,
        eval_dataset    = eval_dataset,
        dataset_text_field = "text",
        max_seq_length  = MAX_SEQ_LENGTH,
        dataset_num_proc = 4,
        packing         = False,     # set True to pack short samples → faster but harder to debug
        args            = training_args,
    )

    # ── 5. Train ─────────────────────────────────────────────────────────────
    print("\nStarting SFT training...")
    trainer_stats = trainer.train()
    print(f"\nTraining complete. Loss: {trainer_stats.training_loss:.4f}")

    # ── 6. Save LoRA adapter ─────────────────────────────────────────────────
    lora_path = os.path.join(OUTPUT_DIR, "lora_adapter")
    model.save_pretrained(lora_path)
    tokenizer.save_pretrained(lora_path)
    print(f"\nLoRA adapter saved to: {lora_path}")

    # ── 7. Test inference ────────────────────────────────────────────────────
    _test_inference(model, tokenizer)

    return model, tokenizer

## MODE B: GRPO — Reinforcement Learning with Verifiable Rewards

In [ ]:
def run_grpo():
    """
    GRPO fine-tuning for vulnerability-aware code fixing.

    Reward rubric (multi-signal, additive):
      +2.0  — response contains a ```java ... ``` fenced code block
      +1.0  — fixed code contains NO obvious vulnerability patterns
      +0.5  — fixed code is shorter than or equal to vulnerable code (concise fix)
      +1.5  — fixed code structurally differs from vulnerable code (actual change)
      -1.0  — response is empty or contains only a code block with no content
      -2.0  — fixed code still contains known vulnerability API calls
    """
    print("\n" + "="*60)
    print("MODE: GRPO (Reinforcement Learning)")
    print("="*60)

    # ── 1. Load model with fast_inference=True (uses vLLM internally) ────────
    from unsloth import FastLanguageModel
    from unsloth.chat_templates import get_chat_template

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name     = MODEL_NAME,
        max_seq_length = MAX_SEQ_LENGTH,
        dtype          = None,
        load_in_4bit   = LOAD_IN_4BIT,
        fast_inference = True,          # enables vLLM-based generation inside GRPO
    )

    tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

    # ── 2. LoRA adapters ─────────────────────────────────────────────────────
    model = FastLanguageModel.get_peft_model(
        model,
        r                          = LORA_RANK,
        target_modules             = TARGET_MODULES,
        lora_alpha                 = LORA_ALPHA,
        lora_dropout               = 0,          # 0 is recommended for GRPO
        bias                       = "none",
        use_gradient_checkpointing = "unsloth",
        random_state               = 42,
        use_rslora                 = True,
    )

    # ── 3. Load dataset — GRPO needs {"prompt": [...messages...]} format ─────
    from datasets import Dataset

    records = []
    with open(DATASET_PATH, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            obj = json.loads(line)
            msgs = obj.get("messages", [])
            # GRPO prompt = system + user turns only (assistant is what the model generates)
            prompt_msgs = [m for m in msgs if m["role"] != "assistant"]
            # Store the ground-truth fix for the reward function
            gt_fix = next((m["content"] for m in msgs if m["role"] == "assistant"), "")
            records.append({
                "prompt":    prompt_msgs,
                "gt_fix":    gt_fix,
            })

    dataset = Dataset.from_list(records)
    print(f"GRPO dataset: {len(dataset)} samples")

    # ── 4. Reward functions ───────────────────────────────────────────────────
    # Known vulnerability API patterns in Java (used for negative reward)
    VULN_PATTERNS = [
        r'Runtime\.getRuntime\(\)\.exec\(',        # command injection
        r'Statement\.execute\s*\(',                  # unparameterized SQL
        r'\.executeQuery\s*\(\s*["\']',              # SQL injection risk
        r'new\s+File\s*\(\s*request\.',              # path traversal
        r'DocumentBuilderFactory\.newInstance\(\)',  # XXE (if not configured)
        r'XPathFactory\.newInstance\(\)',            # XPath injection
        r'ObjectInputStream\s*\(',                   # insecure deserialization
        r'MessageDigest\.getInstance\s*\(\s*["\']MD5', # weak hash
        r'MessageDigest\.getInstance\s*\(\s*["\']SHA-1', # weak hash
        r'new\s+Random\s*\(',                        # non-secure random
    ]

    SAFE_PATTERNS = [
        r'PreparedStatement',           # parameterized SQL
        r'setString\s*\(\d+',           # prepared stmt binding
        r'ProcessBuilder',              # safer process exec
        r'\.toPath\(\)\.normalize\(',   # path normalization
        r'SecureRandom',                # secure random
        r'MessageDigest\.getInstance\s*\(\s*["\']SHA-256', # strong hash
        r'XMLInputFactory',             # configurable XML parser
    ]

    def _extract_java_block(text: str) -> str:
        """Extract content inside the first ```java ... ``` block."""
        m = re.search(r'```java\s*([\s\S]+?)```', text)
        return m.group(1).strip() if m else ""

    def reward_has_code_block(completions, **kwargs) -> list[float]:
        """Reward: +2.0 if response contains a ```java``` fenced block."""
        return [2.0 if "```java" in c and "```" in c[c.index("```java")+7:] else -1.0
                for c in completions]

    def reward_no_vuln_patterns(completions, **kwargs) -> list[float]:
        """Reward: +1.0 if extracted fixed code has no known vulnerability patterns."""
        scores = []
        for c in completions:
            code = _extract_java_block(c)
            if not code:
                scores.append(0.0)
                continue
            hits = sum(1 for p in VULN_PATTERNS if re.search(p, code))
            scores.append(-float(hits))   # -1 per vulnerability pattern found
        return scores

    def reward_uses_safe_patterns(completions, **kwargs) -> list[float]:
        """Reward: +0.5 per safe API usage found in the fix."""
        scores = []
        for c in completions:
            code = _extract_java_block(c)
            if not code:
                scores.append(0.0)
                continue
            hits = sum(0.5 for p in SAFE_PATTERNS if re.search(p, code))
            scores.append(min(hits, 2.0))   # cap at +2.0
        return scores

    def reward_code_changed(completions, prompts, **kwargs) -> list[float]:
        """
        Reward: +1.5 if the fixed code actually differs from the vulnerable code
        in the prompt. Penalises the model for returning identical code.
        """
        scores = []
        for c, prompt_msgs in zip(completions, prompts):
            fixed = _extract_java_block(c)
            if not fixed:
                scores.append(0.0)
                continue
            # Extract original vulnerable code from user prompt
            user_msg = next((m["content"] for m in prompt_msgs if m["role"] == "user"), "")
            vuln = _extract_java_block(user_msg)
            if vuln and fixed.strip() == vuln.strip():
                scores.append(-1.5)   # code unchanged — penalise
            elif fixed:
                scores.append(1.5)
            else:
                scores.append(0.0)
        return scores

    def reward_format(completions, **kwargs) -> list[float]:
        """Reward: +0.5 if response starts with ``` and ends cleanly."""
        return [0.5 if c.strip().startswith("```java") else 0.0 for c in completions]

    reward_functions = [
        reward_has_code_block,
        reward_no_vuln_patterns,
        reward_uses_safe_patterns,
        reward_code_changed,
        reward_format,
    ]

    # ── 5. GRPO Trainer ───────────────────────────────────────────────────────
    from trl import GRPOConfig, GRPOTrainer
    from unsloth import is_bfloat16_supported

    grpo_config = GRPOConfig(
        use_vllm                   = True,          # fast vLLM generation
        learning_rate              = GRPO_LR,
        adam_beta1                 = 0.9,
        adam_beta2                 = 0.99,
        weight_decay               = 0.1,
        warmup_ratio               = 0.1,
        lr_scheduler_type          = "cosine",
        optim                      = "adamw_8bit",
        per_device_train_batch_size = GRPO_BATCH_SIZE,
        gradient_accumulation_steps = GRPO_GRAD_ACCUM,
        num_generations            = GRPO_NUM_GENERATIONS,
        max_prompt_length          = MAX_SEQ_LENGTH // 2,
        max_completion_length      = GRPO_MAX_NEW_TOKENS,
        max_steps                  = GRPO_MAX_STEPS,
        save_steps                 = 50,
        save_total_limit           = 3,
        output_dir                 = OUTPUT_DIR + "_grpo",
        fp16                       = not is_bfloat16_supported(),
        bf16                       = is_bfloat16_supported(),
        report_to                  = "none",
        logging_steps              = 1,
        # GRPO-specific
        temperature                = GRPO_TEMPERATURE,
        epsilon                    = 0.2,
        loss_type                  = "grpo",
        mask_truncated_completions = True,
    )

    trainer = GRPOTrainer(
        model            = model,
        processing_class = tokenizer,
        reward_funcs     = reward_functions,
        args             = grpo_config,
        train_dataset    = dataset,
    )

    # ── 6. Train ─────────────────────────────────────────────────────────────
    print("\nStarting GRPO training... (wait at least 300 steps for reward to rise)")
    trainer.train()

    # ── 7. Save ──────────────────────────────────────────────────────────────
    lora_path = os.path.join(OUTPUT_DIR + "_grpo", "lora_adapter")
    model.save_pretrained(lora_path)
    tokenizer.save_pretrained(lora_path)
    print(f"\nLoRA adapter saved to: {lora_path}")

    _test_inference(model, tokenizer)

    return model, tokenizer

## Quick inference test

In [ ]:
def _test_inference(model, tokenizer):
    """Run a quick inference pass with a hardcoded example to sanity-check the model."""
    from unsloth import FastLanguageModel

    FastLanguageModel.for_inference(model)   # enables 2× faster Unsloth inference

    test_prompt = [
        {
            "role": "system",
            "content": (
                "You are an expert Java security engineer specialising in vulnerability remediation. "
                "When given a security vulnerability report and vulnerable Java code, produce a fixed, "
                "secure version inside a ```java ... ``` block."
            ),
        },
        {
            "role": "user",
            "content": (
                "## Security Vulnerability Report\n\n"
                "**CVE ID:** CVE-2021-EXAMPLE\n"
                "**Vulnerability Type:** SQL Injection\n\n"
                "## Vulnerable Java Code\n\n"
                "```java\n"
                "public List<User> getUser(String username) {\n"
                "    String query = \"SELECT * FROM users WHERE username = '\" + username + \"'\";\n"
                "    return jdbcTemplate.query(query, new UserRowMapper());\n"
                "}\n"
                "```\n\n"
                "Please provide the fixed, secure version of this code."
            ),
        },
    ]

    inputs = tokenizer.apply_chat_template(
        test_prompt,
        tokenize              = True,
        add_generation_prompt = True,
        return_tensors        = "pt",
    ).to("cuda")

    import torch
    with torch.no_grad():
        outputs = model.generate(
            input_ids      = inputs,
            max_new_tokens = 512,
            temperature    = 0.3,
            do_sample      = True,
            pad_token_id   = tokenizer.eos_token_id,
        )

    generated = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)
    print("\n" + "─"*60)
    print("TEST INFERENCE OUTPUT:")
    print("─"*60)
    print(generated)
    print("─"*60)

## Save merged model (optional — for vLLM deployment)

In [ ]:
def save_merged(model, tokenizer, output_dir: str, quantize: str = "f16"):
    """
    Merge LoRA back into base model and save.
    quantize: "f16" | "q4_k_m" | "q8_0"  (GGUF quant type for llama.cpp)
              "f16" means just save as float16 HF model (use with vLLM).
    """
    merged_dir = output_dir + "_merged"

    if quantize == "f16":
        # Save as float16 for vLLM
        model.save_pretrained_merged(merged_dir, tokenizer, save_method="merged_16bit")
        print(f"Merged float16 model saved to: {merged_dir}")
    else:
        # Save as GGUF (for Ollama / llama.cpp)
        model.save_pretrained_gguf(merged_dir, tokenizer, quantization_method=quantize)
        print(f"GGUF ({quantize}) saved to: {merged_dir}")

    return merged_dir

## Push to HuggingFace Hub (optional)

In [ ]:
def push_to_hub(model, tokenizer, repo_name: str, hf_token: str):
    """Push the LoRA adapter to HuggingFace Hub."""
    model.push_to_hub(repo_name, token=hf_token)
    tokenizer.push_to_hub(repo_name, token=hf_token)
    print(f"Pushed to HuggingFace Hub: {repo_name}")

## Main

In [ ]:
# ── Run training ───────────────────────────────────────────────────────────────
from pathlib import Path as _Path

if not _Path(DATASET_PATH).exists():
    print(f"[ERROR] Dataset not found: {DATASET_PATH}")
    print("        Run the 02_clean_and_convert notebook first.")
else:
    if TRAINING_MODE == "sft":
        trained_model, trained_tok = run_sft()
    elif TRAINING_MODE == "grpo":
        trained_model, trained_tok = run_grpo()
    else:
        raise ValueError(f"Unknown TRAINING_MODE: {TRAINING_MODE!r}. Use 'sft' or 'grpo'.")

    if SAVE_MERGED:
        save_merged(trained_model, trained_tok, OUTPUT_DIR)

    if PUSH_TO_HUB:
        import os as _os
        hf_token = _os.environ.get("HF_TOKEN", "")
        if not hf_token:
            print("[WARN] HF_TOKEN env var not set. Hub push skipped.")
        else:
            push_to_hub(trained_model, trained_tok, PUSH_TO_HUB, hf_token)

    print("\n✓ Training complete!")
    print(f"  Adapter saved in: {OUTPUT_DIR}/")
    if SAVE_MERGED:
        print(f"  Merged model:     {OUTPUT_DIR}_merged/")
        print(f"  Serve with vLLM:  vllm serve ./{OUTPUT_DIR}_merged --port 8000")

## Next Steps

- **Serve with vLLM** (set `SAVE_MERGED = True` and re-run the training cell first):
  ```bash
  vllm serve ./output/qwen3-8b-java-vuln-lora_merged --port 8000
  ```
- **Chat with the model**: open and run the `vllm-chat` notebook